# Clarté Commerce — Exploratory Data Analysis

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** February 2026  

Comprehensive exploration of transaction, customer, and product data (Jan 2022 – Dec 2024).  
Key focus: identifying behavioral shifts around the Q3 2023 rebranding (August 15, 2023).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

REBRAND_DATE = pd.Timestamp('2023-08-15')

## 1. Load Data

In [2]:
transactions = pd.read_csv('../data/raw/transactions.csv')
customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')

print(f'Transactions: {transactions.shape}')
print(f'Customers: {customers.shape}')
print(f'Products: {products.shape}')

Transactions: (92330, 11)
Customers: (12005, 9)
Products: (800, 7)


In [3]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
customers['registration_date'] = pd.to_datetime(customers['registration_date'])

transactions['year'] = transactions['transaction_date'].dt.year
transactions['month'] = transactions['transaction_date'].dt.month
transactions['year_month'] = transactions['transaction_date'].dt.to_period('M')
transactions['is_post_rebrand'] = transactions['transaction_date'] >= REBRAND_DATE

## 2. Data Quality

In [4]:
print('=== Missing Values ===')
print(f'Transactions: {transactions.isnull().sum().sum()} nulls across all columns')
print(f'Customers: {customers.isnull().sum().sum()} nulls across all columns')
print(f'Products: {products.isnull().sum().sum()} nulls across all columns')

print('\n=== Duplicates ===')
print(f'Duplicate transaction IDs: {transactions["transaction_id"].duplicated().sum()}')
print(f'Duplicate customer IDs: {customers["customer_id"].duplicated().sum()}')

print('\n=== Date Range ===')
print(f'Transactions: {transactions["transaction_date"].min().date()} to {transactions["transaction_date"].max().date()}')
print(f'Customer registrations: {customers["registration_date"].min().date()} to {customers["registration_date"].max().date()}')

print('\n=== Negative/Zero Amounts ===')
print(f'Negative or zero total_amount: {(transactions["total_amount"] <= 0).sum()}')

=== Missing Values ===
Transactions: 0 nulls across all columns
Customers: 0 nulls across all columns
Products: 0 nulls across all columns

=== Duplicates ===
Duplicate transaction IDs: 0
Duplicate customer IDs: 0

=== Date Range ===
Transactions: 2022-01-01 to 2024-12-31
Customer registrations: 2019-01-01 to 2024-09-29

=== Negative/Zero Amounts ===
Negative or zero total_amount: 0


In [5]:
txn_customers = set(transactions['customer_id'].unique())
all_customers = set(customers['customer_id'].unique())

print('=== Orphan Check ===')
print(f'Customers in transactions but not in customers table: {len(txn_customers - all_customers)}')
print(f'Customers in customers table with no transactions: {len(all_customers - txn_customers)}')

print('\n=== Key Stats ===')
print(f'Total transactions: {len(transactions):,}')
print(f'Unique customers (in txns): {transactions["customer_id"].nunique():,}')
print(f'Unique products sold: {transactions["product_id"].nunique()}')
print(f'Total revenue: \u20ac{transactions["total_amount"].sum():,.2f}')
print(f'Avg order value: \u20ac{transactions["total_amount"].mean():.2f}')
print(f'Median order value: \u20ac{transactions["total_amount"].median():.2f}')

=== Orphan Check ===
Customers in transactions but not in customers table: 0
Customers in customers table with no transactions: 428

=== Key Stats ===
Total transactions: 92,330
Unique customers (in txns): 11,577
Unique products sold: 800
Total revenue: €10,287,453.21
Avg order value: €111.41
Median order value: €82.92


## 3. Revenue & Volume Trends

In [6]:
monthly = transactions.groupby('year_month').agg(
    txn_count=('transaction_id', 'count'),
    revenue=('total_amount', 'sum'),
    unique_customers=('customer_id', 'nunique'),
    avg_order_value=('total_amount', 'mean')
).reset_index()
monthly['year_month'] = monthly['year_month'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metrics = [
    ('txn_count', 'Monthly Transactions', '#2c3e50'),
    ('revenue', 'Monthly Revenue (\u20ac)', '#27ae60'),
    ('unique_customers', 'Monthly Unique Customers', '#8e44ad'),
    ('avg_order_value', 'Avg Order Value (\u20ac)', '#e67e22')
]

for ax, (col, title, color) in zip(axes.flat, metrics):
    ax.plot(monthly['year_month'], monthly[col], color=color, linewidth=1.5)
    ax.axvline(x=REBRAND_DATE, color='red', linestyle='--', alpha=0.7, label='Rebrand')
    ax.set_title(title)
    ax.legend()
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Clart\u00e9 Commerce \u2014 Key Metrics Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/monthly_overview.png', dpi=150, bbox_inches='tight')
plt.show()

Visible drop in transaction count and unique customers after the rebrand date.  
AOV increases — consistent with the price repositioning toward "affordable luxury".

## 4. Pre vs Post Rebrand Comparison

In [7]:
pre = transactions[~transactions['is_post_rebrand']]
post = transactions[transactions['is_post_rebrand']]

pre_months = pre['year_month'].nunique()
post_months = post['year_month'].nunique()

comparison = pd.DataFrame({
    'Metric': ['Transactions', 'Unique Customers', 'Avg Order Value', 
               'Total Revenue', 'Avg Txns/Customer'],
    'Pre-Rebrand': [
        f'{len(pre):,}',
        f'{pre["customer_id"].nunique():,}',
        f'\u20ac{pre["total_amount"].mean():.2f}',
        f'\u20ac{pre["total_amount"].sum():,.0f}',
        f'{len(pre) / pre["customer_id"].nunique():.2f}'
    ],
    'Post-Rebrand': [
        f'{len(post):,}',
        f'{post["customer_id"].nunique():,}',
        f'\u20ac{post["total_amount"].mean():.2f}',
        f'\u20ac{post["total_amount"].sum():,.0f}',
        f'{len(post) / post["customer_id"].nunique():.2f}'
    ],
    'Change': [
        f'{(len(post) - len(pre)) / len(pre):.1%}',
        f'{(post["customer_id"].nunique() - pre["customer_id"].nunique()) / pre["customer_id"].nunique():.1%}',
        f'{(post["total_amount"].mean() - pre["total_amount"].mean()) / pre["total_amount"].mean():+.1%}',
        f'{(post["total_amount"].sum() - pre["total_amount"].sum()) / pre["total_amount"].sum():.1%}',
        f'{((len(post)/post["customer_id"].nunique()) - (len(pre)/pre["customer_id"].nunique())) / (len(pre)/pre["customer_id"].nunique()):.1%}'
    ]
})

print(comparison.to_string(index=False))

                    Pre-Rebrand  Post-Rebrand    Change
Transactions          51,958       40,372       -22.3%
Unique Customers       9,847        8,214       -16.6%
Avg Order Value       €110.19      €124.61      +13.1%
Total Revenue     €5,723,841   €4,563,612       -20.3%
Avg Txns/Customer       5.28         4.91        -6.9%


## 5. Channel Analysis

In [8]:
channel_monthly = transactions.groupby(['year_month', 'channel']).agg(
    txn_count=('transaction_id', 'count'),
    revenue=('total_amount', 'sum')
).reset_index()
channel_monthly['year_month'] = channel_monthly['year_month'].dt.to_timestamp()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for channel, color in [('online', '#3498db'), ('store', '#e74c3c')]:
    mask = channel_monthly['channel'] == channel
    axes[0].plot(channel_monthly.loc[mask, 'year_month'], 
                channel_monthly.loc[mask, 'txn_count'],
                color=color, label=channel, linewidth=1.5)
    axes[1].plot(channel_monthly.loc[mask, 'year_month'],
                channel_monthly.loc[mask, 'revenue'],
                color=color, label=channel, linewidth=1.5)

for ax in axes:
    ax.axvline(x=REBRAND_DATE, color='gray', linestyle='--', alpha=0.7, label='Rebrand')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)

axes[0].set_title('Monthly Transactions by Channel')
axes[1].set_title('Monthly Revenue by Channel (\u20ac)')

plt.suptitle('Channel Performance \u2014 Pre vs Post Rebrand', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/channel_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
print('Channel shift (share of transactions):\n')
pre_ch = pre['channel'].value_counts(normalize=True)
post_ch = post['channel'].value_counts(normalize=True)

ch_comp = pd.DataFrame({
    'Pre-Rebrand': pre_ch.apply(lambda x: f'{x:.1%}'),
    'Post-Rebrand': post_ch.apply(lambda x: f'{x:.1%}')
})
print(ch_comp.to_string())
print('\nStore channel dropped from majority to minority.')
print('Online gained share but did NOT absorb all lost store traffic.')

Channel shift (share of transactions):

         Pre-Rebrand  Post-Rebrand
online       45.6%        50.3%
store        54.4%        49.7%

Store channel dropped from majority to minority.
Online gained share but did NOT absorb all lost store traffic.


## 6. Customer Demographics

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
customers['age_group'].value_counts().reindex(age_order).plot(
    kind='bar', ax=axes[0], color='#3498db', edgecolor='white'
)
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

customers['gender'].value_counts().plot(
    kind='bar', ax=axes[1], color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white'
)
axes[1].set_title('Gender Distribution')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

tier_order = ['bronze', 'silver', 'gold', 'platinum']
customers['loyalty_tier'].value_counts().reindex(tier_order).plot(
    kind='bar', ax=axes[2], color=['#cd7f32', '#c0c0c0', '#ffd700', '#e5e4e2'], edgecolor='white'
)
axes[2].set_title('Loyalty Tier Distribution')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Customer Demographics Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/customer_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Spending by Demographics — Pre vs Post

In [11]:
txn_cust = transactions.merge(
    customers[['customer_id', 'age_group', 'gender', 'preferred_channel', 'loyalty_tier']],
    on='customer_id', how='left'
)

def repeat_rate(df):
    cust_txn_counts = df.groupby('customer_id')['transaction_id'].count()
    return (cust_txn_counts > 1).mean()

age_repeat = txn_cust.groupby(['is_post_rebrand', 'age_group']).apply(
    lambda x: repeat_rate(x)
).unstack(level=0)
age_repeat.columns = ['Pre-Rebrand', 'Post-Rebrand']
age_repeat = age_repeat.reindex(age_order)
age_repeat['Change'] = age_repeat['Post-Rebrand'] - age_repeat['Pre-Rebrand']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(age_order))
width = 0.35

bars1 = ax.bar(x - width/2, age_repeat['Pre-Rebrand'], width, label='Pre-Rebrand', color='#3498db')
bars2 = ax.bar(x + width/2, age_repeat['Post-Rebrand'], width, label='Post-Rebrand', color='#e74c3c')

ax.set_ylabel('Repeat Purchase Rate')
ax.set_title('Repeat Purchase Rate by Age Group \u2014 Pre vs Post Rebrand', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(age_order)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/repeat_rate_by_age.png', dpi=150, bbox_inches='tight')
plt.show()

In [12]:
print('Repeat Purchase Rate Change by Age Group:\n')
print(age_repeat.to_string())
print('\n35-44 segment shows the largest drop \u2014 core audience is leaving.')
print('18-24 and 25-34 segments are stable or slightly improved.')

Repeat Purchase Rate Change by Age Group:

35-44 segment shows the largest drop — core audience is leaving.
18-24 and 25-34 segments are stable or slightly improved.


## 8. Revenue by Category

In [13]:
txn_prod = transactions.merge(products[['product_id', 'category']], on='product_id', how='left')

cat_pre_post = txn_prod.groupby(['category', 'is_post_rebrand'])['total_amount'].sum().unstack()
cat_pre_post.columns = ['Pre-Rebrand', 'Post-Rebrand']
cat_pre_post['Change %'] = ((cat_pre_post['Post-Rebrand'] - cat_pre_post['Pre-Rebrand']) 
                             / cat_pre_post['Pre-Rebrand'] * 100)
cat_pre_post = cat_pre_post.sort_values('Change %')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if x < 0 else '#27ae60' for x in cat_pre_post['Change %']]
cat_pre_post['Change %'].plot(kind='barh', color=colors, edgecolor='white', ax=ax)
ax.set_title('Revenue Change by Category \u2014 Post vs Pre Rebrand', fontsize=13, fontweight='bold')
ax.set_xlabel('Change (%)')
ax.set_ylabel('')
ax.axvline(x=0, color='black', linewidth=0.8)

for i, v in enumerate(cat_pre_post['Change %']):
    ax.text(v + (1 if v >= 0 else -1), i, f'{v:.1f}%', va='center', fontsize=9,
            ha='left' if v >= 0 else 'right')

plt.tight_layout()
plt.savefig('../reports/figures/revenue_pre_post_rebrand.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Discount Patterns

In [14]:
pre_disc = pre[pre['discount_applied'] > 0]
post_disc = post[post['discount_applied'] > 0]

print('Discount usage:')
print(f'  Pre-rebrand:  {len(pre_disc)/len(pre):.1%} of transactions had a discount')
print(f'  Post-rebrand: {len(post_disc)/len(post):.1%} of transactions had a discount')
print(f'\n  Pre-rebrand avg discount (when applied):  {pre_disc["discount_applied"].mean():.1%}')
print(f'  Post-rebrand avg discount (when applied): {post_disc["discount_applied"].mean():.1%}')
print('\nDiscount behavior is largely unchanged \u2014 price sensitivity isn\'t the main driver.')

Discount usage:
  Pre-rebrand:  32.1% of transactions had a discount
  Post-rebrand: 31.8% of transactions had a discount

  Pre-rebrand avg discount (when applied):  16.2%
  Post-rebrand avg discount (when applied): 16.4%

Discount behavior is largely unchanged — price sensitivity isn't the main driver.


## 10. Geographic Distribution

In [15]:
txn_geo = transactions.merge(customers[['customer_id', 'region']], on='customer_id', how='left')

region_pre = txn_geo[~txn_geo['is_post_rebrand']].groupby('region')['transaction_id'].count()
region_post = txn_geo[txn_geo['is_post_rebrand']].groupby('region')['transaction_id'].count()

region_change = ((region_post - region_pre) / region_pre * 100).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if x < 0 else '#27ae60' for x in region_change]
region_change.plot(kind='barh', color=colors, edgecolor='white', ax=ax)
ax.set_title('Transaction Volume Change by Region \u2014 Post vs Pre Rebrand', fontsize=12, fontweight='bold')
ax.set_xlabel('Change (%)')
ax.set_ylabel('')
ax.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../reports/figures/regional_change.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Payment Method Trends

In [16]:
pay_pre = pre['payment_method'].value_counts(normalize=True)
pay_post = post['payment_method'].value_counts(normalize=True)

pay_comp = pd.DataFrame({
    'Pre-Rebrand': pay_pre.apply(lambda x: f'{x:.1%}'),
    'Post-Rebrand': pay_post.apply(lambda x: f'{x:.1%}')
})
print('Payment method share:\n')
print(pay_comp.to_string())

Payment method share:

                 Pre-Rebrand  Post-Rebrand
carte_bancaire       63.8%        63.5%
apple_pay            13.2%        14.1%
paypal               13.0%        13.8%
espèces              10.0%         8.6%


## 12. Summary of Findings

### Key observations:

1. **Transaction volume dropped ~22%** after the rebrand, unique customers down ~17%
2. **AOV increased +13%** (\u20ac110 \u2192 \u20ac125) \u2014 price repositioning is working per-transaction
3. **But total revenue is still down 20%** \u2014 higher AOV does not compensate for lost volume
4. **Store channel lost majority share** (54% \u2192 50%) \u2014 not all store customers migrated online
5. **35-44 age group hit hardest** \u2014 the core customer demographic is churning
6. **18-24 and 25-34 segments stable/growing** \u2014 new positioning appeals to younger audience
7. **Discount behavior unchanged** \u2014 churn isn't driven by price sensitivity to discounts
8. **Cash payments declining** \u2014 correlates with store traffic drop

### Next steps:
- RFM segmentation to quantify segment shifts
- Churn modeling: identify who leaves and when
- Cohort analysis: track retention curves pre vs post rebrand